# Probe anatomy figures

The LEC equivalent of the figures in
[AdamLoydHarris/probe_tracing](https://github.com/AdamLoydHarris/probe_tracing)
(El-Gaby et al. 2024, PFC dataset), rebuilt on this project's corrected histology.

| figure | old repo | here |
|---|---|---|
| recording sites coloured by region | `plot_probes_with_anatomical_locations` | `pf.plot_channel_regions` |
| probe tracts in the brain (3D) | `plot_experiment_probe_tracts` | `pf.plot_probe_tracts` |
| mouse colour key | the `__main__` block | `pf.plot_mouse_legend` |

**Run in the `histology` conda env.** The 3D figure needs a display; on a headless
node launch jupyter (or the script) under `xvfb-run -a`.

The old pipeline's `get_probe_anatomy_df.py` (HERBS → AllenSDK → per-site acronym)
is not needed: `code/histology_refit/` already produces that table from hand-corrected
fits. This notebook is figures only — it changes no fit and no anatomy.

In [ ]:
import sys; sys.path.insert(0, '.')
import matplotlib.pyplot as plt
import pandas as pd

import probe_figures as pf

print('mice     :', pf.MICE)
print('figures  ->', pf.FIGURE_DIR)

## 1. Gates — run these before believing any figure

Two things can go silently wrong in a figure like this, and both are invisible by eye:

1. **Scrambled shanks.** The old code reshaped a flat array to approximate probe
   geometry. Ours pivots on measured `(x, y)`, and asserts every one of the 384
   contacts lands in exactly one cell of the 48×8 grid. Shanks sit in *different
   structures*, so a scramble would produce a plausible-looking, wrong figure.
2. **A region with no colour**, silently dropped. Ported from the old repo's assert.

`verify` additionally checks the grid's composition and the units panel against
`channel_regions.csv` / `unit_regions.csv`, so the figure has to agree with the census.

In [ ]:
print(pf.check_palette_complete())
gates = pf.verify()

## 2. Recording sites by brain region

One column-block per mouse — 4 shanks, 2 contact columns each, 48 depths, tip at the
bottom — with every site coloured by its Allen structure. Layers are shaded within
family (ENTl as one ramp, ENTm as another), which is what makes the layer sequence
visible; `✗` marks contacts never recorded in any block.

The right-hand column of each pair is the **units panel**: QC single units per depth,
stacked by region. This is the thing the PFC pipeline could not do, and it answers a
different question from the site strip — yield varies by region, so where the silicon
sat and where the cells were are not the same picture.

In [ ]:
pf.plot_channel_regions(save=True); plt.show()

## 3. Mouse colour key

Defines the colours used for the 3D tracts. Kept as its own file so the same key can
sit beside any panel.

In [ ]:
pf.plot_mouse_legend(save=True); plt.show()

## 4. Probe tracts in the brain (3D)

brainrender scene: brain surface, ENTl/ENTm/SUB/ProS/CA1 meshes, and one line per
shank per mouse. The full modelled shank is drawn thin and the recorded bank thick,
so trajectory and recording extent are both visible.

Coordinates come from `allen_atlas_coords`, which are **already in µm** in the atlas's
`asr` frame — none of the rescaling or origin-flipping the HERBS pipeline needed.
Computing them needs the ~7 GB deformation fields, so they are cached to disk beside
each fit and keyed on a hash of that fit: correct the fit and the cache rebuilds itself.

Views: `sagittal` (left lateral — the side the probes are in), `sagittal2` (the other
hemisphere), `top`, and two three-quarter views 90° apart. Any preset can be rotated
with a `_rot<deg>` suffix.

Saved as **PNG**. `vector=True` also writes PDF/SVG, but vedo's GL2PS exporter
tessellates every triangle of the brain mesh and takes many minutes — figures 2 and 3
above are matplotlib and are always vector.

In [ ]:
scene, paths = pf.plot_probe_tracts(save=True)
for p in paths:
    print(p.name)

### Recording sites only

The same scene with the insertion track removed entirely — just the contacts that
actually recorded (377–383 per mouse, dead channels dropped), drawn as points. This is
the honest picture of where data came from, as opposed to where the probe went. The
brain is made more transparent and the points are unlit so the mouse colours survive
being viewed through the surface.


In [ ]:
scene, paths = pf.plot_probe_tracts(sites_only=True, save=True)
for p in paths:
    print(p.name)

### Interactive

With a display available, this opens the scene so it can be spun by hand — useful for
picking a camera before committing to a saved view.

In [ ]:
# scene, _ = pf.plot_probe_tracts(save=False, interactive=True)

## 5. Caveats that belong in any caption

- **Shank identity is an exact degeneracy.** Mirroring the probe reproduces contact
  positions to 0.0 µm, so which physical shank carries channels 0–95 is not resolvable
  from the dye — only from the surgical record. `SHANK_ORDER_VERIFIED` is still `False`,
  hence the `?` on shank numbers. The *ordering* is internally consistent (shank 0 is
  anterior-most in all five mice) and anatomy at a physical position is invariant, so
  the region assignments are unaffected — but if the array was mirrored, read the
  columns right-to-left.
- **Registration precision.** The local deformation field runs 8.06–12.95 µm per voxel,
  so structure boundaries carry roughly that much positional uncertainty.
- **ly05 has the weakest anatomy** in the cohort — its DiI was unusable (167 points,
  97 µm span), so its fit is a hand placement carrying `fit_disagrees_with_dye_axis`,
  `low_signal_coverage` and `high_residual` flags.
- **Units are unit-recordings, not unique neurons**: each mouse's 5 blocks are
  independent sorts of the same probe in the same brain.
- ah09 has histology but was never recorded, so it is excluded; pass
  `mice=(..., 'ah09')` to include it in the 3D scene.